# Exp0.1.3 — Doc-faithful SAE-Dense Regularization
Analysis-only notebook for the 30 binary direct-SNN regularized runs paired against frozen Exp0.1 controls.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('artifacts/experiment_0_1_3_doc_faithful_regularization/doc_faithful_sae_dense_binary_v1')
manifest = json.loads((ROOT / 'manifest.json').read_text())
runs = pd.read_csv(ROOT / 'runs.csv')
summary = pd.read_csv(ROOT / 'summary.csv')
comparison = pd.read_csv(ROOT / 'comparison_summary.csv')
paired = pd.read_csv(ROOT / 'paired_regularization_effects.csv')
objective_interactions = pd.read_csv(ROOT / 'regularization_objective_interactions.csv')
architecture_interactions = pd.read_csv(ROOT / 'regularization_architecture_interactions.csv')
manifest


## Five-seed performance summary


In [ ]:
display(summary.sort_values(['architecture', 'objective', 'regularization']))
display(comparison.sort_values('mean_test_ba', ascending=False))


## Paired regularization effect


In [ ]:
effect = (paired.groupby(['architecture', 'objective'])
          .agg(mean_delta_ba=('delta_test_ba_doc_reg_minus_off', 'mean'),
               sd_delta_ba=('delta_test_ba_doc_reg_minus_off', 'std'),
               mean_delta_event_rate=('delta_last_hidden_events_per_neuron_second', 'mean'),
               mean_dead_fraction=('on_test_mean_dead_neuron_fraction', 'mean'))
          .reset_index())
display(effect)
ax = effect.pivot(index='architecture', columns='objective', values='mean_delta_ba').plot(kind='bar')
ax.axhline(0, linewidth=1)
ax.set_ylabel('Mean paired test BA delta (doc reg - off)')
plt.tight_layout()


## Objective and architecture interactions


In [ ]:
display(objective_interactions.groupby('architecture').mean(numeric_only=True))
display(architecture_interactions.groupby(['comparison', 'objective']).mean(numeric_only=True))


## Training-scale diagnostics
Read existing history CSVs only. Check whether weighted P2/A1 dominate the task objective and which P2 layer contributes most.


In [ ]:
history_files = sorted((ROOT / 'histories').glob('*.csv'))
history_rows = []
for path in history_files:
    h = pd.read_csv(path)
    for which, row in [('first', h.iloc[0]), ('last', h.iloc[-1])]:
        item = {'run': path.stem, 'point': which}
        for col in h.columns:
            if col in {'train_task_loss', 'train_p2', 'train_a1', 'train_weighted_p2', 'train_weighted_a1', 'train_reg_to_task_ratio', 'train_p2_output'} or col.startswith('train_p2_hidden_'):
                item[col] = row[col]
        history_rows.append(item)
history_scale = pd.DataFrame(history_rows)
display(history_scale)
display(history_scale.groupby('point').mean(numeric_only=True))
